# CMS Dataset Variable Exploration

This notebook queries a small sample from the public CMS dataset to inspect its structure before formal analysis. Run the cells from top to bottom to retrieve the sample, list the variables, and inspect the first row. The sample remains in memory and is not saved locally.

In [ ]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen

URL = "https://data.cms.gov/data-api/v1/dataset/690ddc6c-2767-4618-b277-420ffb2bf27c/data"
SAMPLE_SIZE = 10  # Change this value to inspect a different sample size.

query_url = f"{URL}?{urlencode({'size': SAMPLE_SIZE})}"
with urlopen(query_url, timeout=30) as response:
    records = json.load(response)

if not records:
    raise ValueError("The API returned no records.")

print(f"Loaded {len(records)} rows.")

## Variables Returned by the CMS API

In [ ]:
columns = list(records[0].keys())

print(f"Total columns: {len(columns)}")
for number, column in enumerate(columns, start=1):
    print(f"{number}. {column}")

## First Sample Row

In [ ]:
first_row = records[0]

for column, value in first_row.items():
    print(f"{column}: {value}")

# Dimensions and Measures

A **dimension** identifies, describes, or categorizes an observation. A **measure** represents a quantitative value that can be meaningfully aggregated, compared, or summarized. A variable can contain numbers and still be a dimension if those numbers function as identifiers or categories.

The roles below use the official CMS data dictionary and are limited to variables observed in the API sample.

| Variable | Role | Reason |
| -------- | ---- | ------ |
| `Rndrng_Prvdr_CCN` | Identifier | CMS Certification Number identifies the rendering hospital provider; it should remain text to preserve leading zeros. |
| `Rndrng_Prvdr_Org_Name` | Descriptive attribute | Human-readable organization name associated with the provider CCN. |
| `Rndrng_Prvdr_City` | Dimension | Categorizes the provider by its reported city. |
| `Rndrng_Prvdr_St` | Descriptive attribute | Reported provider street address; useful for description, not aggregation. |
| `Rndrng_Prvdr_State_FIPS` | Dimension | Standard state geographic code; numeric characters represent a category, not a quantity. |
| `Rndrng_Prvdr_Zip5` | Dimension | Five-digit postal geography used for grouping or filtering, not arithmetic. |
| `Rndrng_Prvdr_State_Abrvtn` | Dimension | Readable state category for the provider location. |
| `Rndrng_Prvdr_RUCA` | Dimension | Rural-Urban Commuting Area code is a categorical geographic classification. |
| `Rndrng_Prvdr_RUCA_Desc` | Descriptive attribute | Human-readable label for the RUCA code. |
| `DRG_Cd` | Identifier | Identifies the MS-DRG service category within the reporting year. |
| `DRG_Desc` | Descriptive attribute | Human-readable clinical and severity label associated with the DRG code. |
| `Tot_Dschrgs` | Measure | Count of reported inpatient discharges for the provider and DRG. |
| `Avg_Submtd_Cvrd_Chrg` | Measure | Average submitted covered charge for the provider and DRG. |
| `Avg_Tot_Pymt_Amt` | Measure | Average total payment amount for the provider and DRG. |
| `Avg_Mdcr_Pymt_Amt` | Measure | Average Medicare payment amount for the provider and DRG. |

# Primary Key Hypothesis

A primary key uniquely identifies each row in a table. The apparent dataset grain is **one hospital + one DRG**, so the initial candidate key combines the provider identifier and the service-category identifier.

> **Hypothesis:** the combination of provider CCN and DRG code uniquely identifies each record in the current 2024 provider-and-service dataset.

This is a hypothesis, not a confirmed fact. The test below covers only the API sample currently loaded in memory.

## Test: Composite-Key Uniqueness in the Current Sample

The test compares the sample row count with the number of distinct `Rndrng_Prvdr_CCN` + `DRG_Cd` combinations and inspects any duplicated combinations.

In [ ]:
import pandas as pd

sample_df = pd.DataFrame(records)
key_columns = ["Rndrng_Prvdr_CCN", "DRG_Cd"]

total_row_count = len(sample_df)
unique_key_count = sample_df[key_columns].drop_duplicates().shape[0]
duplicate_mask = sample_df.duplicated(subset=key_columns, keep=False)
duplicate_rows = sample_df.loc[duplicate_mask].sort_values(key_columns)
duplicate_combination_count = duplicate_rows[key_columns].drop_duplicates().shape[0]

print("Validation scope: current API sample only")
print(f"Total rows: {total_row_count}")
print(f"Unique provider + DRG combinations: {unique_key_count}")
print(f"Duplicate combinations: {duplicate_combination_count}")

if duplicate_rows.empty:
    print("Result: no duplicate key combinations were found in this sample.")
else:
    print("Result: duplicate key combinations were found in this sample.")
    display(duplicate_rows)

## Result and Interpretation

**Sample result:** the current 10-row API sample contains 10 unique provider-DRG combinations and no duplicate combinations.

**Interpretation:** the result is consistent with the candidate key, but it does not validate uniqueness across the complete 2024 dataset. The hypothesis remains open until the full dataset is acquired and tested.

## Current Data Grain Hypothesis

> Based on the CMS documentation and the current API sample, one row appears to represent one hospital-provider and one DRG combination for the selected reporting year. This grain will be formally validated once the complete dataset is acquired.